In [2]:
pip install requests beautifulsoup4 pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


#DATA COLLECTION#

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# domains you want to collect
DOMAINS = {
    "Criminal": "https://indiankanoon.org/search/?formInput=criminal",
    "Civil": "https://indiankanoon.org/search/?formInput=civil",
    "Constitutional": "https://indiankanoon.org/search/?formInput=constitution",
    "Family": "https://indiankanoon.org/search/?formInput=divorce+OR+marriage",
    "Labour": "https://indiankanoon.org/search/?formInput=labour+employment",
    "Taxation": "https://indiankanoon.org/search/?formInput=income+tax+OR+gst",
    "Corporate": "https://indiankanoon.org/search/?formInput=company+law+OR+IBC",
    "IPR": "https://indiankanoon.org/search/?formInput=patent+OR+copyright+OR+trademark",
    "Environmental": "https://indiankanoon.org/search/?formInput=environment+pollution",
}

def get_case_links(base_url, max_pages=2):
    """Fetch case links from Indian Kanoon search pages"""
    links = []
    for page in range(max_pages):
        url = f"{base_url}&pagenum={page}"
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")

        for a in soup.select("a"):
            href = a.get("href")
            if href and href.startswith("/doc/"):  # case doc link
                links.append("https://indiankanoon.org" + href)

        time.sleep(2)  # be polite, avoid IP ban
    return list(set(links))

def scrape_case(url):
    """Scrape case text from a single Indian Kanoon case"""
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    # title
    title = soup.find("h1").get_text(strip=True) if soup.find("h1") else ""

    # judgment text (inside div class 'judgments')
    content = " ".join([p.get_text(" ", strip=True) for p in soup.select("div.judgments p")])
    return title, content

# Collect data per domain
all_data = []
for domain, search_url in DOMAINS.items():
    print(f"Collecting {domain} cases...")
    case_links = get_case_links(search_url, max_pages=3)  # adjust pages

    for link in case_links[:50]:  # limit per domain for demo
        try:
            title, text = scrape_case(link)
            all_data.append({"domain": domain, "title": title, "text": text, "url": link})
        except Exception as e:
            print("Error:", e)
        time.sleep(1)        

Error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [4]:
# Save raw dataset
df = pd.DataFrame(all_data)
df.to_csv("indiankanoon_raw.csv", index=False)
print("Saved dataset with", len(df), "cases")

Saved dataset with 303 cases


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Function to fetch cases for one domain
def fetch_cases(domain, pages_to_scrape=5):
    base_url = "https://indiankanoon.org/search/?formInput="
    cases = []

    print(f"🔹 Fetching cases from {domain.title()} domain...")

    for page in range(0, pages_to_scrape * 20, 10):
        url = f"{base_url}{domain}+doctypes%3Ajudgments&pagenum={page//10 + 1}"
        try:
            response = requests.get(url, timeout=15)
            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.text, "html.parser")
            results = soup.find_all("div", class_="result_title")

            for res in results:
                case_link = "https://indiankanoon.org" + res.a["href"]
                case_title = res.get_text(strip=True)
                cases.append({"title": case_title, "link": case_link})

        except Exception as e:
            print(f"⚠️ Error fetching page {page//10 + 1} of {domain}: {e}")
            continue

        time.sleep(1)  # gentle pause to avoid server blocking

    # Save cases to CSV
    df = pd.DataFrame(cases)
    df.to_csv(f"{domain}_cases.csv", index=False, encoding="utf-8")

    print(f"✅ Successfully fetched {len(df)} cases from {domain.title()} domain.")
    return df


# 9 Legal Domains (balanced dataset)
domains = [
    "constitutional",
    "criminal",
    "civil",
    "corporate",
    "environmental",
    "family",
    "labour",
    "property",
    "taxation"
]

# Fetch and save data for each domain
all_data = []

for domain in domains:
    df = fetch_cases(domain, pages_to_scrape=25)
    all_data.append(df)

# Combine all domains into one CSV
combined = pd.concat(all_data, ignore_index=True)
combined.to_csv("all_domains_cases.csv", index=False, encoding="utf-8")

print(f"\n🎉 Completed scraping for all domains! Total cases collected: {len(combined)}")

🔹 Fetching cases from Constitutional domain...
⚠️ Error fetching page 31 of constitutional: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ Successfully fetched 380 cases from Constitutional domain.
🔹 Fetching cases from Criminal domain...
✅ Successfully fetched 390 cases from Criminal domain.
🔹 Fetching cases from Civil domain...
✅ Successfully fetched 390 cases from Civil domain.
🔹 Fetching cases from Corporate domain...
✅ Successfully fetched 390 cases from Corporate domain.
🔹 Fetching cases from Environmental domain...
✅ Successfully fetched 390 cases from Environmental domain.
🔹 Fetching cases from Family domain...
✅ Successfully fetched 390 cases from Family domain.
🔹 Fetching cases from Labour domain...
✅ Successfully fetched 390 cases from Labour domain.
🔹 Fetching cases from Property domain...
✅ Successfully fetched 390 cases from Property domain.
🔹 Fetching cases from Taxation domain...
✅ Successfully fetched 390 cases from Taxat

In [6]:
import pandas as pd
import glob
import os

# Find all individual domain CSV files
files = glob.glob("*_cases.csv")

dfs = []

for file in files:
    # Extract the domain name from the filename (e.g., "criminal_cases.csv" → "criminal")
    domain_name = os.path.basename(file).replace("_cases.csv", "")
    
    # Read CSV and add domain column
    df = pd.read_csv(file)
    df["domain"] = domain_name
    dfs.append(df)

# Combine all domain DataFrames
combined = pd.concat(dfs, ignore_index=True)

# Save final combined CSV
combined.to_csv("all_domains_cases.csv", index=False, encoding="utf-8")

print(f"✅ Combined {len(files)} domain files into all_domains_cases.csv")
print(f"📄 Total cases combined: {len(combined)}")
print("📁 Added 'domain' column for labeled dataset")

✅ Combined 10 domain files into all_domains_cases.csv
📄 Total cases combined: 7000
📁 Added 'domain' column for labeled dataset


In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os

# Read the combined CSV
all_cases = pd.read_csv("all_domains_cases.csv")

# Add a new column for full text if it doesn't exist
if "judgment_text" not in all_cases.columns:
    all_cases["judgment_text"] = ""

# Function to fetch full text of a case
def get_full_text(url):
    try:
        response = requests.get(url, timeout=20)
        if response.status_code != 200:
            return ""
        soup = BeautifulSoup(response.text, "html.parser")
        paragraphs = soup.select("div.judgments p")
        text = " ".join([p.get_text(" ", strip=True) for p in paragraphs])
        return text
    except:
        return ""

# Scrape full text for each case
for idx, row in all_cases.iterrows():
    # Skip if text already exists (resume-friendly)
    if row["judgment_text"].strip():
        continue

    all_cases.at[idx, "judgment_text"] = get_full_text(row["link"])
    print(f"Fetched case {idx+1}/{len(all_cases)}: {row['domain']} - {row['title'][:50]}...")

    time.sleep(1)  # polite delay per case

    # Save progress every 50 cases
    if (idx + 1) % 50 == 0:
        all_cases.to_csv("all_domains_cases_fulltext.csv", index=False, encoding="utf-8")
        print(f"💾 Progress saved at case {idx+1}")

# Final save
all_cases.to_csv("all_domains_cases_fulltext.csv", index=False, encoding="utf-8")
print(f"🎉 All cases updated with full judgment text! Total cases: {len(all_cases)}")

Fetched case 1/7000: all_domains - Maneka Gandhi vs Union Of India on 25 January, 197...
Fetched case 2/7000: all_domains - Olga Tellis & Ors vs Bombay Municipal Corporation ...
Fetched case 3/7000: all_domains - Union Of India And Another vs Tulsiram Patel And O...
Fetched case 4/7000: all_domains - A.K. Gopalan vs The State Of Madras.Union Of India...
Fetched case 5/7000: all_domains - Janata Dal vs H.S. Chowdhary And Ors. on 28 August...
Fetched case 6/7000: all_domains - Secretary, State Of Karnataka And ... vs Umadevi A...
Fetched case 7/7000: all_domains - Ajay Hasia Etc vs Khalid Mujib Sehravardi & Ors. E...
Fetched case 8/7000: all_domains - The State Of Bihar vs Maharajadhiraja Sir Kameshwa...
Fetched case 9/7000: all_domains - Central Inland Water ... vs Brojo Nath Ganguly & A...
Fetched case 10/7000: all_domains - Balco Employees Union (Regd.) vs Union Of India & ...
Fetched case 11/7000: all_domains - Tofan Singh vs The State Of Tamil Nadu on 29 Octob...
Fetched case 12/700

In [8]:
import pandas as pd

# Load full-text CSV
df = pd.read_csv("all_domains_cases_fulltext.csv")

# Quick cleaning: remove newlines and extra spaces
df["judgment_text"] = df["judgment_text"].str.replace("\n", " ").str.strip()

# Optional: remove rows with empty judgment_text
df = df[df["judgment_text"].str.len() > 50]  # keep cases with text > 50 chars
print(f"✅ Total usable cases: {len(df)}")

✅ Total usable cases: 505


#data preprocessing#

In [9]:
df = pd.read_csv("all_domains_cases_fulltext.csv")  # replace with your 316-case file

# Keep only usable cases (non-empty judgment_text)
usable_df = df[df["judgment_text"].str.strip().str.len() > 50]

# Count number of cases per domain
domain_counts = usable_df["domain"].value_counts()

print("📊 Number of usable cases per domain (316 cases):")
print(domain_counts)


📊 Number of usable cases per domain (316 cases):
domain
all_domains      250
taxation         181
civil             27
corporate         21
criminal           7
family             7
environmental      5
property           5
labour             2
Name: count, dtype: int64


In [10]:
pip install datasets transformers


Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd

df = pd.read_csv("all_domains_cases_fulltext.csv")
print("📄 Columns in your CSV:", df.columns.tolist())
print("\n🔹 Sample rows:")
print(df.head(3))


📄 Columns in your CSV: ['title', 'link', 'domain', 'judgment_text']

🔹 Sample rows:
                                               title  \
0  Maneka Gandhi vs Union Of India on 25 January,...   
1  Olga Tellis & Ors vs Bombay Municipal Corporat...   
2  Union Of India And Another vs Tulsiram Patel A...   

                                                link       domain  \
0  https://indiankanoon.org/docfragment/1766147/?...  all_domains   
1  https://indiankanoon.org/docfragment/709776/?f...  all_domains   
2  https://indiankanoon.org/docfragment/1134697/?...  all_domains   

  judgment_text  
0           NaN  
1           NaN  
2           NaN  


In [12]:
import pandas as pd

df = pd.read_csv("all_domains_cases_fulltext.csv")

# Drop rows where text is missing
df = df.dropna(subset=['judgment_text'])

print("✅ Non-empty judgments:", len(df))
print("\n📊 Domain distribution:")
print(df['domain'].value_counts())


✅ Non-empty judgments: 505

📊 Domain distribution:
domain
all_domains      250
taxation         181
civil             27
corporate         21
criminal           7
family             7
environmental      5
property           5
labour             2
Name: count, dtype: int64


In [13]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

# Load and clean
df = pd.read_csv("all_domains_cases_fulltext.csv")
df = df.dropna(subset=['judgment_text', 'domain'])

# Remove any domain with <3 samples
domain_counts = df['domain'].value_counts()
rare_domains = domain_counts[domain_counts < 3].index
df = df[~df['domain'].isin(rare_domains)]

print("✅ Kept domains with ≥3 samples")
print(df['domain'].value_counts())

# Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['domain'])

# Split data safely
if df['label'].value_counts().min() < 2:
    print("⚠️ Not enough data to stratify — using random split.")
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
else:
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["judgment_text"], truncation=True, padding="max_length", max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("\n✅ Data ready for model training!")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

✅ Kept domains with ≥3 samples
domain
all_domains      250
taxation         181
civil             27
corporate         21
criminal           7
family             7
environmental      5
property           5
Name: count, dtype: int64


Map: 100%|██████████| 51/51 [00:00<00:00, 171.75 examples/s]


✅ Data ready for model training!
Train samples: 402
Val samples: 50
Test samples: 51
Label mapping: {'all_domains': np.int64(0), 'civil': np.int64(1), 'corporate': np.int64(2), 'criminal': np.int64(3), 'environmental': np.int64(4), 'family': np.int64(5), 'property': np.int64(6), 'taxation': np.int64(7)}


In [14]:
pip install hf_xet

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("all_domains_cases_fulltext.csv")
df = df.dropna(subset=['judgment_text', 'domain'])

# Remove rare domains (<3 samples)
domain_counts = df['domain'].value_counts()
rare_domains = domain_counts[domain_counts < 3].index
df = df[~df['domain'].isin(rare_domains)]

# Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['domain'])
num_labels = len(label_encoder.classes_)
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


Label mapping: {'all_domains': np.int64(0), 'civil': np.int64(1), 'corporate': np.int64(2), 'criminal': np.int64(3), 'environmental': np.int64(4), 'family': np.int64(5), 'property': np.int64(6), 'taxation': np.int64(7)}


In [16]:
# Previous mapping:
# {'all_domains': 0, 'civil': 1, 'corporate': 2, 'criminal': 3, 'environmental': 4, 'family': 5, 'property': 6, 'taxation': 7}
num_labels = 8

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
pip install --upgrade transformers

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.0
    Uninstalling transformers-4.57.0:
      Successfully uninstalled transformers-4.57.0
Note: you may need to restart the kernel to use updated packages.


In [18]:
import sys
print(sys.executable)

c:\Users\Navya\anaconda3\envs\legal_nlp\python.exe


In [19]:
import transformers
print(transformers.__file__)
print(transformers.__version__)


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\transformers\__init__.py
4.57.0


In [20]:
from transformers import TrainingArguments
print(TrainingArguments)
print(TrainingArguments.__module__)


<class 'transformers.training_args.TrainingArguments'>
transformers.training_args


In [21]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

In [22]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [23]:
pip install accelerate>=0.26.0

Note: you may need to restart the kernel to use updated packages.


In [24]:
from transformers import TrainingArguments

try:
    training_args = TrainingArguments(
        output_dir="./outputs",
        num_train_epochs=10,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        logging_steps=20,
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )
    print("TrainingArguments initialized successfully.")
except ImportError as e:
    print("ImportError:", e)
except Exception as e:
    print("Error:", e)


TrainingArguments initialized successfully.


In [25]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [26]:
pip install evaluate

Note: you may need to restart the kernel to use updated packages.


In [27]:
from transformers import Trainer
import evaluate
import numpy as np

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")

# Define the evaluation metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 🚀 Start training
trainer.train()

C:\Users\Navya\AppData\Local\Temp\ipykernel_17476\1784777584.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy
50,1.253000,1.388870,0.480000
100,1.199900,1.280352,0.360000
150,1.190500,1.275567,0.360000
200,0.979600,1.292799,0.280000
250,0.974300,1.285385,0.260000
300,0.938600,1.373286,0.300000
350,0.842900,1.438222,0.240000
400,0.828000,1.537921,0.200000
450,0.813600,1.558071,0.200000
500,0.850500,1.597652,0.180000


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Navya\anaconda3\envs\legal_nlp\

TrainOutput(global_step=510, training_loss=1.0077219766729018, metrics={'train_runtime': 11611.4454, 'train_samples_per_second': 0.346, 'train_steps_per_second': 0.044, 'total_flos': 1057763422863360.0, 'train_loss': 1.0077219766729018, 'epoch': 10.0})

In [28]:
# Evaluate on test set
test_results = trainer.evaluate(test_dataset)
print("\n✅ Test Set Evaluation Complete!")
print(test_results)


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



✅ Test Set Evaluation Complete!
{'eval_loss': 1.2807763814926147, 'eval_accuracy': 0.2549019607843137, 'eval_runtime': 24.7101, 'eval_samples_per_second': 2.064, 'eval_steps_per_second': 0.162, 'epoch': 10.0}


In [31]:
import numpy as np
import pandas as pd

# Make predictions
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# Convert label indices back to domain names
true_domains = label_encoder.inverse_transform(predictions.label_ids)
pred_domains = label_encoder.inverse_transform(pred_labels)

# Create a DataFrame for comparison
results_df = pd.DataFrame({
    "True Domain": true_domains,
    "Predicted Domain": pred_domains
})

# Calculate per-domain accuracy
domain_accuracy = results_df.groupby("True Domain").apply(lambda x: (x["True Domain"] == x["Predicted Domain"]).mean())
print("\n📊 Per-domain accuracy:\n", domain_accuracy)

# Optionally view a few misclassifications
print("\n🔍 Sample misclassifications:")
print(results_df[results_df["True Domain"] != results_df["Predicted Domain"]].head(10))

c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



📊 Per-domain accuracy:
 True Domain
all_domains    0.269231
civil          0.000000
corporate      0.000000
criminal       0.000000
family         0.000000
taxation       0.300000
dtype: float64

🔍 Sample misclassifications:
    True Domain Predicted Domain
1      criminal      all_domains
2   all_domains         taxation
3      taxation      all_domains
4   all_domains         taxation
6      taxation      all_domains
7         civil      all_domains
10  all_domains         taxation
11  all_domains         taxation
12  all_domains         taxation
13     taxation      all_domains


C:\Users\Navya\AppData\Local\Temp\ipykernel_17476\3445561975.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  domain_accuracy = results_df.groupby("True Domain").apply(lambda x: (x["True Domain"] == x["Predicted Domain"]).mean())


In [16]:
from sklearn.utils import resample

df_majority = df[df['domain'] == 'all_domains']
df_minority = df[df['domain'] != 'all_domains']

df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=min(len(df_majority), len(df_minority)),
    random_state=42
)


df_balanced = pd.concat([df_majority_downsampled, df_minority])


In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(df["domain"])  # or df_balanced["domain"], depending on which DataFrame you're using


LabelEncoder()

In [11]:
print(df.columns)


Index(['title', 'link', 'domain', 'judgment_text'], dtype='object')


In [12]:
# Encode your labels and create a 'label' column in the dataframe
df['label'] = label_encoder.fit_transform(df['domain'])


In [13]:
import torch
from torch.nn import CrossEntropyLoss

# Compute class weights inversely proportional to class frequency
class_weights = torch.tensor(
    [1 / df['label'].value_counts()[i] for i in range(len(label_encoder.classes_))],
    dtype=torch.float
)

loss_fn = CrossEntropyLoss(weight=class_weights)


In [15]:
import pandas as pd
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
import torch
from torch.nn import CrossEntropyLoss

# Load your dataset
df = pd.read_csv("all_domains_cases_fulltext.csv")

# Keep only relevant columns
df = df.dropna(subset=["judgment_text", "domain"])

from sklearn.utils import resample

# Identify majority and minority classes dynamically
domain_counts = df['domain'].value_counts()
majority_class = domain_counts.idxmax()
minority_class = domain_counts.idxmin()

df_majority = df[df['domain'] == majority_class]
df_minority = df[df['domain'] == minority_class]

# Downsample majority to match minority
df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

df_balanced = pd.concat([df_majority_downsampled, df_minority])


df_balanced = pd.concat([df_majority_downsampled, df_minority])

# Encode labels
label_encoder = LabelEncoder()
df_balanced["label"] = label_encoder.fit_transform(df_balanced["domain"])

# Class-weighted loss
class_weights = torch.tensor(
    [1 / df_balanced['label'].value_counts()[i] for i in range(len(label_encoder.classes_))]
)
loss_fn = CrossEntropyLoss(weight=class_weights)

print("✅ df_balanced recreated with shape:", df_balanced.shape)
print("📊 Domain distribution:\n", df_balanced['domain'].value_counts())


✅ df_balanced recreated with shape: (4, 5)
📊 Domain distribution:
 domain
all_domains    2
labour         2
Name: count, dtype: int64


In [17]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Count samples per domain
domain_counts = df_balanced['domain'].value_counts()

# Keep only domains that have >= 2 cases
valid_domains = domain_counts[domain_counts >= 2].index
df_balanced = df_balanced[df_balanced['domain'].isin(valid_domains)]

print("✅ Domains with >= 2 cases retained:")
print(df_balanced['domain'].value_counts())

# Encode labels again
label_encoder = LabelEncoder()
df_balanced["label"] = label_encoder.fit_transform(df_balanced["domain"])

# Train-validation split (now safe)
train_df, val_df = train_test_split(
    df_balanced,
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["label"]
)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df[["judgment_text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["judgment_text", "label"]])

print(f"✅ Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")


✅ Domains with >= 2 cases retained:
domain
all_domains      250
taxation         181
civil             27
corporate         21
criminal           7
family             7
environmental      5
property           5
labour             2
Name: count, dtype: int64
✅ Training samples: 404 | Validation samples: 101


In [ ]:
pip install -U transformers


Note: you may need to restart the kernel to use updated packages.


In [18]:
print(df_balanced.columns)


Index(['title', 'link', 'domain', 'judgment_text', 'label'], dtype='object')


In [ ]:
import transformers
print(transformers.__version__)


4.57.0


In [29]:
!pip uninstall transformers -y
!pip install transformers==4.57.0



Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
  Using cached transformers-4.57.0-py3-none-any.whl.metadata (41 kB)
Using cached transformers-4.57.0-py3-none-any.whl (12.0 MB)


Reason for being yanked: Error in the setup causing installation issues


In [28]:
from transformers import TrainingArguments
help(TrainingArguments)


Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: float = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType

In [27]:
import transformers
print(transformers.__file__)



c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\transformers\__init__.py


In [19]:
print(train_dataset.column_names)
print(val_dataset.column_names)


['judgment_text', 'label', '__index_level_0__']
['judgment_text', 'label', '__index_level_0__']


In [20]:
def tokenize_function(example):
    return tokenizer(example["judgement_text"], padding="max_length", truncation=True, max_length=512)


In [37]:
def tokenize_function(example):
    return tokenizer(example["judgement_text"], padding="max_length", truncation=True, max_length=1024)



In [38]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")

def tokenize_function(example):
    return tokenizer(example["judgment_text"], padding="max_length", truncation=True, max_length=1024)


In [39]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


Map: 100%|██████████| 101/101 [00:00<00:00, 117.85 examples/s]


In [40]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
import torch
import numpy as np

# -----------------------------
# Step 1: Tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")

def tokenize_function(example):
    return tokenizer(example["judgment_text"], padding="max_length", truncation=True, max_length=512)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# -----------------------------
# Step 2: Ensure labels column exists
# -----------------------------
# Copy 'label' to 'labels' for Trainer compatibility
train_dataset = train_dataset.map(lambda x: {"labels": x["label"]})
val_dataset = val_dataset.map(lambda x: {"labels": x["label"]})

# -----------------------------
# Step 3: Load Model
# -----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    "law-ai/InLegalBERT",
    num_labels=len(np.unique(train_dataset["label"]))
)

# -----------------------------
# Step 4: Training Arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results_inlegalbert",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs_inlegalbert",
    logging_steps=50,
    save_total_limit=2,
    eval_strategy="steps",  # ✅ works in 4.57
    eval_steps=50
)


# -----------------------------
# Step 5: Custom Trainer (Weighted Loss)
# -----------------------------
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Automatically compute class weights
        classes = np.unique(train_dataset["label"])
        counts = np.bincount(train_dataset["label"])
        weight = torch.tensor([len(train_dataset)/counts[i] for i in classes], dtype=torch.float).to(logits.device)

        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# -----------------------------
# Step 6: Trainer Initialization
# -----------------------------
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

# -----------------------------
# Step 7: Train the model
# -----------------------------
trainer.train()


Map: 100%|██████████| 101/101 [00:00<00:00, 3996.12 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at law-ai/InLegalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Navya\AppData\Local\Temp\ipykernel_21404\3732155101.py:73: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
50,2.102400,1.899785
100,1.621800,1.874700
150,1.630100,1.847994
200,1.501600,1.780201
250,1.353300,1.802527
300,1.500700,1.682539
350,1.373400,1.664282
400,1.296300,1.706388
450,1.177600,1.746023
500,1.348000,2.019733


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1010, training_loss=1.287367562020179, metrics={'train_runtime': 12421.9819, 'train_samples_per_second': 0.325, 'train_steps_per_second': 0.081, 'total_flos': 1063035471421440.0, 'train_loss': 1.287367562020179, 'epoch': 10.0})

In [41]:
results = trainer.evaluate()
print(results)


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 2.186776638031006, 'eval_runtime': 51.6377, 'eval_samples_per_second': 1.956, 'eval_steps_per_second': 0.252, 'epoch': 10.0}


In [42]:
trainer.save_model("inlegal_bert")
tokenizer.save_pretrained("inlegal_bert")


('inlegal_bert\\tokenizer_config.json',
 'inlegal_bert\\special_tokens_map.json',
 'inlegal_bert\\vocab.txt',
 'inlegal_bert\\added_tokens.json',
 'inlegal_bert\\tokenizer.json')

In [43]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="inlegal_bert",
    tokenizer="inlegal_bert"
)

example_text = "The company violated the contract terms."
print(classifier(example_text))



Device set to use cpu


[{'label': 'LABEL_0', 'score': 0.33082395792007446}]


In [44]:
# Define the class names in the same order as your labels (0–7)
label_names = [
    "Civil", "Corporate", "Family", "Taxation",
    "Criminal", "Labour", "Property", "Other"
]


In [45]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="legal_bert_finetuned",
    tokenizer="legal_bert_finetuned",
    device=-1  # CPU; use device=0 for GPU
)

example_text = "The company violated the contract terms."
result = classifier(example_text)[0]

# Map LABEL_# to class name
label_index = int(result['label'].split("_")[-1])
result['label'] = label_names[label_index]

print(result)


Device set to use cpu


{'label': 'Other', 'score': 0.17183460295200348}


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = r"C:\Users\Navya\Desktop\navya lap\College\SEM5\NLP\Project\inlegal_bert"
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print(model.config.id2label)


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8'}


In [3]:
from transformers import AutoModelForSequenceClassification
import os

old_path = r"C:\Users\Navya\Desktop\navya lap\College\SEM5\NLP\Project\inlegal_bert"
new_path = os.path.join(os.path.dirname(old_path), "inlegal_bert_updated")

model = AutoModelForSequenceClassification.from_pretrained(old_path)

model.config.id2label = {
    0: "Civil",
    1: "Corporate",
    2: "Family",
    3: "Taxation",
    4: "Criminal",
    5: "Labour",
    6: "Property",
    7: "Other",
    8: "Miscellaneous"
}
model.config.label2id = {v: k for k, v in model.config.id2label.items()}

model.save_pretrained(new_path)
print(f"✅ Model label mapping saved successfully to {new_path}")



✅ Model label mapping saved successfully to C:\Users\Navya\Desktop\navya lap\College\SEM5\NLP\Project\inlegal_bert_updated


In [1]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained(r"C:\Users\Navya\Desktop\navya lap\College\SEM5\NLP\Project\inlegal_bert")
print(config.id2label)


c:\Users\Navya\anaconda3\envs\legal_nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{0: 'Civil', 1: 'Corporate', 2: 'Family', 3: 'Taxation', 4: 'Criminal', 5: 'Labour', 6: 'Property', 7: 'Other', 8: 'Miscellaneous'}
